# Kratt — BERT authenticity classifier

Trains a weighted BERT on `labeled_comments.csv` (the output of the LLM-labeling notebook) to
score comments as **authentic vs bot**.

**The weighting idea (why BERT + weights):** `niche_tag` is the PRIMARY signal and `comment_tag`
the SECONDARY one. The same comment type means different things in different niches -- a
low-effort comment under a genuine-niche video (tutorial, stunt) is usually just a casual human,
so it gets a **high** authenticity score; the same low-effort comment under a low-effort-niche
video (fast-consume clips) matches the bot pattern and gets a **low** score. That niche x tag
matrix produces a per-comment authenticity score, which becomes BOTH the training label
(authentic if >= 0.5) AND a per-sample **loss weight** (confident combinations pull training
harder; ambiguous ones barely count). Sample-weighted fine-tuning is exactly where BERT-style
models earn their keep over fixed-rule classifiers.

**Niche + engagement go INTO the input (critical).** The label depends on `niche_tag` -- a VIDEO
property not in the comment text -- so without it the same comment carries opposite labels across
niches (contradictory signal; earlier runs stuck at ~60% for exactly this). Each input is prefixed
with a **readable** context string, e.g. `niche low effort . few likes . no replies . <comment>`:
- **readable words, not an opaque `<NICHE_X>` token** -- the model already knows 'niche', 'low',
  'effort', 'likes', 'replies', so it uses them immediately instead of learning a new embedding
  from scratch (that opaque token is why the first niche attempt still lost to bag-of-words).
- **like_count / reply_count** are strong authenticity signals (real comments earn engagement,
  bots rarely do) that were previously unused.
All three are known at serve time (video niche + scraped counts), so this is fair. Verified: a
TF-IDF baseline goes 55% (text only) -> 71% with this input, so BERT has a clear path past 70%.

**Evaluation:** classic confusion matrix. Positive class = `bot`, so a **false positive =
flagged as bot but labeled authentic** -- the costly error for a media-literacy tool. The final
cell exports `handcheck_sample.csv` for manual review, because the labels themselves come from
the niche x tag matrix (weak labels): the confusion matrix measures agreement with the matrix,
not absolute truth. Hand-checking is the only way to catch where the matrix itself is wrong.

## Setup
**Hosted notebook (Kaggle):** run this cell once; if it installs anything for the first time
this session, restart the kernel (Run -> Restart Session) before running the rest.

In [ ]:
import os, sys
IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or os.path.exists('/kaggle')

if IN_KAGGLE:
    # Kaggle ships torch/transformers matched to its GPU image -- do NOT reinstall those
    # (doing so into a live kernel causes the torch/Trainer import mismatch seen before).
    !pip install -q langdetect accelerate
else:
    # Local venv: pip install -r ../requirements.txt (plus langdetect)
    pass

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'      # Windows OpenMP double-load guard; no-op on Linux
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'    # hf_transfer stalls silently on large files
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '60')

import faulthandler; faulthandler.enable()
import re, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE,
      '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')

try:
    from transformers import Trainer  # noqa: F401
    print('transformers import OK')
except Exception as e:
    raise RuntimeError(
        'transformers/torch import is broken in this kernel -- restart the session '
        '(Run -> Restart Session) and Run All from the top.') from e

# --- config ---
_candidates = [Path('/kaggle/input/datasets/geraldadli/kratt-comments/labeled_comments.csv')]
if IN_KAGGLE:
    _candidates += sorted(Path('/kaggle/input').rglob('labeled_comments.csv'))
_candidates += [Path('labeled_comments.csv'),                        # local: same folder
                Path(r'C:\Users\Asus\Documents\Big Four\Kratt2\kratt\backend\notebooks\labeled_comments.csv')]
DATA_PATH = next((p for p in _candidates if p.exists()), _candidates[0])
print('data:', DATA_PATH, '| exists:', DATA_PATH.exists())

# Model source. PREFER the canonical hub checkpoint 'xlm-roberta-base': the attached Kaggle copy
# stores LayerNorm affine params under the legacy names gamma/beta, which the current (very new)
# transformers no longer auto-renames to weight/bias -- so those pretrained LayerNorm weights
# silently fail to load (see the 'missing/unexpected keys ... LayerNorm' warning in the earlier
# run) and the backbone starts handicapped. The canonical checkpoint uses the modern names.
# hf_transfer is disabled in Setup, so the download that stalled before should complete now;
# if it fails, we fall back to the local copy (handicapped but runnable). Loader is in the
# training cell -- MODEL_SOURCES lists them in preference order.
_LOCAL_MODEL = Path('/kaggle/input/models/mohammedhamdan/xlm-roberta-base/transformers/default/1/xlm-roberta-base')
MODEL_SOURCES = ['xlm-roberta-base']
if IN_KAGGLE and _LOCAL_MODEL.exists():
    MODEL_SOURCES.append(str(_LOCAL_MODEL))

MAX_LEN    = 128
EPOCHS     = 3
SEED       = 42
OUTPUT_DIR = Path('/kaggle/working/bert_kratt_out') if IN_KAGGLE else Path('bert_kratt_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)

## Load
Expects the labeled export: `comment_id, video_id, niche_tag, text, ..., like_count,
reply_count, comment_tag, label_source`.

In [ ]:
df = pd.read_csv(DATA_PATH)
print('shape:', df.shape)
display(df.head(3))

missing = df['comment_tag'].isna().sum()
if missing:
    print(f'dropping {missing} rows without a comment_tag (never reached by the LLM pass)')
    df = df[df['comment_tag'].notna()].copy()

print('\nniche_tag x comment_tag counts:')
print(pd.crosstab(df['niche_tag'], df['comment_tag']))

## Preprocess
Rules:
1. **Remove non-English comments** (langdetect; emoji-only comments are kept -- no alphabetic
   content means nothing to detect, and emoji are signal).
2. **Keep emojis** -- no demojize, no stripping. Zero-width joiners inside emoji sequences
   (family/skin-tone emoji) are preserved.
3. **Lowercase, EXCEPT all-caps tokens** -- SHOUTING is a signal a cased model can use.
   Invisible characters (zero-width spaces, BOM, soft hyphens, directional marks) are removed.
4. **Relabel entities as tokens**: URLs -> `<URL>`, @mentions -> `<USER>`, #hashtags -> `<TAG>`
   (registered as special tokens with the tokenizer later, so they stay single units).

In [ ]:
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 0   # langdetect is stochastic by default; pin it

# Invisible chars: ZWSP, directional marks, word-joiner family, BOM, soft hyphen.
# Deliberately NOT removed: U+200D (zero-width joiner -- glues family/profession emoji
# sequences together) and U+FE0F (emoji variation selector). 'Keep emojis' means keeping
# the invisible glue they're built from, too.
INVISIBLE_RE = re.compile('[\u200b\u200e\u200f\u2060\u2061\u2062\u2063\u2064\ufeff\u00ad]')
URL_RE  = re.compile(r'(?:https?://|www\.)\S+', re.IGNORECASE)
USER_RE = re.compile(r'@[\w.\-]+')
TAG_RE  = re.compile(r'#\w+')
ALPHA_RE = re.compile(r'[^\W\d_]')   # any Unicode letter

def preprocess(text):
    s = str(text)
    s = INVISIBLE_RE.sub('', s)
    s = URL_RE.sub(' <URL> ', s)    # URLs first: they can contain '#' and '@' fragments
    s = USER_RE.sub(' <USER> ', s)
    s = TAG_RE.sub(' <TAG> ', s)
    toks = []
    for tok in s.split():
        # all-caps tokens (>= 2 chars) keep their case -- shouting is a signal. <URL>/<USER>/
        # <TAG> count as all-caps, so the placeholders survive untouched by construction.
        toks.append(tok if (len(tok) >= 2 and tok.isupper()) else tok.lower())
    return ' '.join(toks)

def is_english(text):
    s = re.sub(r'<URL>|<USER>|<TAG>', ' ', str(text))
    letters = ALPHA_RE.findall(s)
    if not letters:
        return True   # emoji-only / numeric-only -> keep (emoji are kept per the rules)
    ascii_ratio = sum(c.isascii() for c in letters) / len(letters)
    if ascii_ratio < 0.7:
        return False  # mostly non-Latin script -> not English
    if len(s.split()) <= 3:
        # langdetect is unreliable on very short text ('nice video' comes back as Danish...);
        # for short mostly-ASCII comments, keep them and let the model judge
        return ascii_ratio >= 0.9
    try:
        return detect(s) == 'en'
    except LangDetectException:
        return ascii_ratio >= 0.9

t0 = time.time()
df['clean_text'] = df['text'].map(preprocess)
keep = df['clean_text'].map(is_english)
print(f'language filter: keeping {keep.sum()} / {len(df)} comments '
      f'({(~keep).sum()} non-English removed) in {time.time()-t0:.0f}s')
df = df[keep].reset_index(drop=True)
display(df[['text', 'clean_text']].head(8))

## Authenticity weights — niche_tag primary, comment_tag secondary
The matrix below assigns every (niche, comment) combination an authenticity score in [0, 1].
Key encoded beliefs (tune these values as the team's theory sharpens -- they are the product's
actual opinion, so argue about them!):
- `ads_spam` is a bot regardless of niche (0.05 everywhere).
- A `genuine` comment is authentic in any niche (it engages with content).
- **`low-effort` comment in a `genuine` niche -> 0.75 (high authentic)** -- casual human viewers.
- **`low-effort` comment in a `low-effort` niche -> 0.20** -- the bot pattern the team observed.
- `copycat` in a `genuine` niche is a coin flip (0.50) -- humans echo top comments too. Its
  sample weight becomes ~0, so these ambiguous rows barely influence training.

From the score: `label = authentic if score >= 0.5 else bot`, and
`sample_weight = |score - 0.5| * 2` (0 = ambiguous, ~1 = certain).

In [ ]:
NICHES = ['genuine', 'copycat', 'low-effort']
CTAGS  = ['genuine', 'copycat', 'low-effort', 'ads_spam']

AUTHENTICITY = pd.DataFrame(
    #  genuine  copycat  low-effort  ads_spam     <- comment_tag
    [[  0.95,    0.50,     0.75,      0.05 ],     # niche: genuine
     [  0.90,    0.30,     0.45,      0.05 ],     # niche: copycat
     [  0.85,    0.30,     0.20,      0.05 ]],    # niche: low-effort
    index=pd.Index(NICHES, name='niche_tag'),
    columns=pd.Index(CTAGS, name='comment_tag'))
print('authenticity score matrix (rows = niche_tag, cols = comment_tag):')
display(AUTHENTICITY)

df = df[df['niche_tag'].isin(NICHES) & df['comment_tag'].isin(CTAGS)].copy()
df['auth_score'] = df.apply(lambda r: AUTHENTICITY.loc[r['niche_tag'], r['comment_tag']], axis=1)
df['label'] = (df['auth_score'] >= 0.5).astype(int)          # 1 = authentic, 0 = bot
df['sample_weight'] = (df['auth_score'] - 0.5).abs() * 2.0

# Put niche + engagement signals INTO the model input. Three reasons this fixes the stuck-at-60%:
#  1. The label depends on niche_tag (a VIDEO property, not in the comment text) -- without it the
#     model sees identical text with opposite labels across niches (contradictory signal).
#  2. A READABLE prefix ('niche low effort') uses words the model already knows. An opaque
#     <NICHE_X> special token has to learn its embedding from scratch in a few epochs -- that was
#     exactly why the first niche attempt still underperformed a bag-of-words baseline.
#  3. like_count / reply_count are strong authenticity signals (real comments earn likes and
#     replies; bots rarely do) that were previously unused. Bucketed into readable phrases.
# Verified on this dataset: a TF-IDF baseline goes 55% (text only) -> 71% with this exact input.
def like_phrase(n):  return 'many likes' if n >= 10 else ('some likes' if n >= 2 else 'few likes')
def reply_phrase(n): return 'has replies' if n > 0 else 'no replies'
df['model_text'] = ('niche ' + df['niche_tag'].str.replace('-', ' ') + ' . '
                    + df['like_count'].map(like_phrase) + ' . '
                    + df['reply_count'].map(reply_phrase) + ' . ' + df['clean_text'])
print('example model input:', df['model_text'].iloc[0][:100])

id2label = {0: 'bot', 1: 'authentic'}
label2id = {v: k for k, v in id2label.items()}
print('\nlabel counts:'); print(df['label'].map(id2label).value_counts())
print('\nmean sample weight by (niche, tag):')
print(df.groupby(['niche_tag', 'comment_tag'])['sample_weight'].mean().round(2).unstack())

In [ ]:
# --- per-video cap, then split by video_id ---
# Cap BEFORE splitting: without it, one giant comment section can be most of the dataset and
# the grouped split degenerates (an earlier run produced test videos=1 with test BIGGER than
# train -- the confusion matrix measured a single video's comment section, which is meaningless).
MAX_PER_VIDEO = 500
before_rows = len(df)
df = (df.sample(frac=1, random_state=SEED)   # shuffle, then head() = random cap, deterministic
        .groupby('video_id')
        .head(MAX_PER_VIDEO)
        .reset_index(drop=True))
print(f'per-video cap {MAX_PER_VIDEO}: {before_rows} -> {len(df)} rows across '
      f'{df["video_id"].nunique()} videos')

from sklearn.model_selection import GroupShuffleSplit
X = df['model_text'].values   # niche-prefixed text (see the niche-token fix above)
y = df['label'].values
w = df['sample_weight'].values
groups = df['video_id'].values

def find_split(lo, hi):
    # accept a seed only if the split is actually sane: test share in [lo, hi], at least 2
    # distinct videos in test, and both classes present on both sides
    for seed in range(300):
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        _tr, _te = next(gss.split(X, y, groups))
        frac = len(_te) / len(X)
        if (lo <= frac <= hi and df.iloc[_te]['video_id'].nunique() >= 2
                and len(set(y[_tr])) == 2 and len(set(y[_te])) == 2):
            return _tr, _te, seed, frac
    return None

found = find_split(0.15, 0.25)
if found is None:
    found = find_split(0.10, 0.35)
    if found is not None:
        print('WARNING: no seed produced a 15-25% test share; accepted a looser 10-35% split.')
if found is None:
    raise RuntimeError(
        'Could not build a sane grouped split: this file has too few videos (or one video still '
        'dominates after the cap). Lower MAX_PER_VIDEO, or combine more part files before training.')
tr, te, split_seed, frac = found
assert not (set(df.iloc[tr]['video_id']) & set(df.iloc[te]['video_id'])), 'video leaked!'
print(f'split seed {split_seed}: train={len(tr)} test={len(te)} ({frac:.0%} test) | '
      f'test videos={df.iloc[te]["video_id"].nunique()}')
print('test label counts:', {id2label[int(k)]: int(v)
                             for k, v in zip(*np.unique(y[te], return_counts=True))})

## Weighted BERT fine-tune
Two kinds of weights, multiplied in the loss:
- **class weights** -- balance bot vs authentic. Crucially, computed on **effective weighted
  mass** (sum of per-sample weights per class), NOT raw counts. The matrix gives authentic rows
  higher sample weights on average than bot rows, so raw-count balancing let the authentic class
  dominate the loss even though bots are the ~74% majority -- that was the predict-authentic
  bias / high-false-negative failure in the earlier run. `BOT_WEIGHT_MULT` adds a further push
  on the bot class to cut false negatives.
- **per-sample weights** -- the authenticity-matrix confidence, so certain combinations
  (ads_spam: 0.9) teach the model hard while ambiguous ones (copycat-in-genuine: 0.0) are
  effectively ignored.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score
import torch.nn as nn, inspect

# try each source in preference order (canonical hub first, local copy as fallback)
def load_from_sources(loader, sources, **kw):
    last = None
    for src in sources:
        try:
            obj = loader(src, **kw)
            print('loaded from:', src)
            return obj
        except Exception as e:
            print(f'  could not load from {src}: {type(e).__name__}: {str(e)[:120]}')
            last = e
    raise last

tokenizer = load_from_sources(AutoTokenizer.from_pretrained, MODEL_SOURCES)
# niche/likes/replies are readable WORDS in model_text now (not special tokens) -- only the
# entity placeholders need registering so they stay single units.
tokenizer.add_special_tokens({'additional_special_tokens': ['<URL>', '<USER>', '<TAG>']})

_TOKENIZER_KWARG = ('processing_class' if 'processing_class' in
                    inspect.signature(Trainer.__init__).parameters else 'tokenizer')

class CommentDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, weights):
        self.texts, self.labels, self.weights = list(texts), list(labels), list(weights)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN)
        enc['labels'] = int(self.labels[i])
        enc['sample_weight'] = float(self.weights[i])
        return enc

_base_collator = DataCollatorWithPadding(tokenizer)
def collate(features):
    # pull the custom key out before padding -- tokenizer.pad() only guarantees handling of
    # its own model-input keys, and silently dropping sample_weight would break the loss
    weights = torch.tensor([f.pop('sample_weight') for f in features], dtype=torch.float)
    batch = _base_collator(features)
    batch['sample_weight'] = weights
    return batch

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, preds, average='macro'),
            'accuracy': float((preds == p.label_ids).mean())}

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        weights = inputs.pop('sample_weight')
        out = model(**inputs)
        device = next(model.parameters()).device   # model.device breaks under DataParallel
        per_sample = nn.CrossEntropyLoss(weight=self.cw.to(device),
                                          reduction='none')(out.logits, labels)
        weights = weights.to(device)
        loss = (per_sample * weights).sum() / weights.sum().clamp(min=1e-8)
        return (loss, out) if return_outputs else loss

# Class balance on EFFECTIVE weighted mass, not raw counts: the loss multiplies class weight
# by per-sample weight, and authentic rows carry higher sample weights on average, so raw-count
# 'balanced' weights let the authentic class dominate the loss (~1.5x) despite bots being the
# majority -- the exact predict-authentic / high-false-negative bias of the earlier run.
BOT_WEIGHT_MULT = 1.0   # extra push on the bot class. Kept at 1.0 now that niche is IN the input
                        # (the model can actually discriminate, so effective-mass balancing is
                        # enough). Raise toward 1.5 if bots still slip through; lower if FPs climb.
mass = np.array([w[tr][y[tr] == 0].sum(), w[tr][y[tr] == 1].sum()])   # [bot, authentic]
cw = mass.sum() / (2.0 * np.clip(mass, 1e-8, None))
cw[0] *= BOT_WEIGHT_MULT
cw = torch.tensor(cw, dtype=torch.float)
print(f'effective class mass [bot, authentic]: {mass.round(0)} -> class weights {cw.numpy().round(3)}')

model = load_from_sources(AutoModelForSequenceClassification.from_pretrained, MODEL_SOURCES,
                          num_labels=2, id2label=id2label, label2id=label2id)
model.resize_token_embeddings(len(tokenizer))   # room for <URL>/<USER>/<TAG>

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'train'),
    per_device_train_batch_size=16, gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCHS, learning_rate=2e-5, weight_decay=0.01,
    fp16=(DEVICE == 'cuda'),
    eval_strategy='epoch', save_strategy='epoch',   # older transformers: evaluation_strategy
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    logging_steps=100, report_to='none', seed=SEED)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=CommentDS(X[tr], y[tr], w[tr]),
    eval_dataset=CommentDS(X[te], y[te], w[te]),
    data_collator=collate, compute_metrics=compute_metrics, class_weights=cw,
    **{_TOKENIZER_KWARG: tokenizer})
trainer.train()

## Metric — confusion matrix
Positive class = **bot**. So:
- **False positive (top-right of the bot row breakdown): predicted bot, labeled authentic** --
  the error that unjustly flags real people; this is what the manual hand-check targets.
- False negative: a bot that slipped through.

Remember the labels come from the niche x tag matrix -- disagreement can mean the MODEL is
wrong or the MATRIX is wrong. That's what the hand-check sample decides.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

pred_out = trainer.predict(CommentDS(X[te], y[te], w[te]))
probs = torch.softmax(torch.tensor(pred_out.predictions), dim=-1).numpy()
preds = pred_out.predictions.argmax(-1)

print(classification_report(y[te], preds, target_names=['bot', 'authentic']))

cm = confusion_matrix(y[te], preds, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Greens')
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred: bot', 'pred: authentic'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['true: bot', 'true: authentic'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center')
plt.title('confusion matrix (test set)'); plt.tight_layout(); plt.show()

fp = int(cm[1, 0])   # labeled authentic, predicted bot
fn = int(cm[0, 1])   # labeled bot, predicted authentic
print(f'false positives (authentic flagged as bot): {fp}')
print(f'false negatives (bot slipped through):      {fn}')

# threshold sweep: trade false negatives against false positives WITHOUT retraining.
# Default decision is prob_bot >= 0.5; lowering the threshold catches more bots (fewer FNs)
# at the cost of more FPs. Pick the row that matches the product's tolerance.
print('\nbot-flag threshold sweep (flag as bot when prob_bot >= t):')
print('  t   bot recall   bot precision     FP     FN')
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    flag = probs[:, 0] >= t
    tp_  = int(((y[te] == 0) & flag).sum())
    fp_  = int(((y[te] == 1) & flag).sum())
    fn_  = int(((y[te] == 0) & ~flag).sum())
    rec  = tp_ / max(tp_ + fn_, 1)
    prec = tp_ / max(tp_ + fp_, 1)
    print(f' {t:.1f}   {rec:9.1%}   {prec:12.1%}   {fp_:5d}  {fn_:5d}')

In [ ]:
# --- export a manual hand-check sample ---
# Weak labels mean disagreements are ambiguous by construction: model wrong, or matrix wrong?
# Humans decide. Fill the empty human_verdict column with 'bot' / 'authentic' and compare.
test_df = df.iloc[te].copy()
test_df['predicted'] = pd.Series(preds, index=test_df.index).map(id2label)
test_df['prob_bot'] = probs[:, 0]
test_df['weak_label'] = test_df['label'].map(id2label)

fp_rows = test_df[(test_df['weak_label'] == 'authentic') & (test_df['predicted'] == 'bot')]
fn_rows = test_df[(test_df['weak_label'] == 'bot') & (test_df['predicted'] == 'authentic')]
ok_rows = test_df[test_df['weak_label'] == test_df['predicted']]

handcheck = pd.concat([
    fp_rows.sample(n=min(150, len(fp_rows)), random_state=SEED),
    fn_rows.sample(n=min(100, len(fn_rows)), random_state=SEED),
    ok_rows.sample(n=min(50, len(ok_rows)), random_state=SEED),   # control group
])
cols = ['comment_id', 'video_id', 'niche_tag', 'comment_tag', 'auth_score',
        'weak_label', 'predicted', 'prob_bot', 'text']
handcheck = handcheck[cols].copy()
handcheck['human_verdict'] = ''
HANDCHECK_PATH = OUTPUT_DIR / 'handcheck_sample.csv'
handcheck.to_csv(HANDCHECK_PATH, index=False)
print(f'exported {len(handcheck)} rows ({len(fp_rows)} total FPs, {len(fn_rows)} total FNs '
      f'in test set) -> {HANDCHECK_PATH}')

## Save the final model
**The final model lives at `/kaggle/working/bert_kratt_out/best/`.** To take it with you:
1. **Save Version** (top right) and let it run to completion.
2. Open the notebook page -> **Output** tab -> `bert_kratt_out/best/` -- download it, or attach
   it to another notebook directly via *+ Add Input -> Your Work -> this notebook's output*.
3. The backend loads it with
   `AutoModelForSequenceClassification.from_pretrained('.../bert_kratt_out/best')`.

Epoch checkpoints under `bert_kratt_out/train/` are deleted below -- they are several GB of
intermediate state that would bloat the notebook Output for no benefit.

In [ ]:
import shutil

FINAL_MODEL_DIR = OUTPUT_DIR / 'best'   # <<< the one directory you need
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
(FINAL_MODEL_DIR / 'authenticity_matrix.json').write_text(AUTHENTICITY.to_json())
(FINAL_MODEL_DIR / 'labels.json').write_text(json.dumps({'id2label': id2label, 'label2id': label2id}))

# drop the multi-GB epoch checkpoints so the notebook Output stays lean and downloadable
shutil.rmtree(OUTPUT_DIR / 'train', ignore_errors=True)

print('=' * 60)
print('FINAL MODEL SAVED TO:', FINAL_MODEL_DIR.resolve())
print('=' * 60)
for p in sorted(FINAL_MODEL_DIR.iterdir()):
    print(f'  {p.name:35s} {p.stat().st_size / 1e6:9.1f} MB')
print('\nAlso in the Output:', (OUTPUT_DIR / 'handcheck_sample.csv').resolve())

## Next steps
1. **Hand-check** `handcheck_sample.csv` -- fill `human_verdict`, then compute agreement three
   ways: human vs model, human vs matrix label, model vs matrix. If humans side with the model
   against the matrix on some (niche, tag) cell, fix the MATRIX value and retrain -- that's the
   cheap fix.
2. **Tune the threshold**: `prob_bot` is exported; if false positives dominate, require
   `prob_bot > 0.7` (instead of 0.5) to flag a bot.
3. **Serve it**: the backend loads `bert_kratt_out/best`, scores each fetched comment, and
   aggregates mean authenticity into the % breakdown for `/analyze`.